# MAE underwater inpainting v1

Train a small Masked Autoencoder for the 8x8 patch competition setup. The notebook writes `submission_v1.csv` and keeps visible test patches unchanged before any encoding step.


## Imports and configuration

Keep configuration explicit and small. `FAST_DEV_RUN=True` is for a quick smoke run; set it to `False` for a real Kaggle GPU run.


In [ ]:

import csv
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import traceback
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

@dataclass
class Config:
    seed: int = 5364
    version: str = 'v1'
    image_size: int = 256
    patch_size: int = 32
    num_patches: int = 64
    patch_dim: int = 32 * 32 * 3
    min_masked: int = 8
    max_masked: int = 20
    embed_dim: int = 256
    encoder_depth: int = 4
    encoder_heads: int = 4
    decoder_dim: int = 128
    decoder_depth: int = 2
    decoder_heads: int = 4
    vq_classes: int = 1024
    batch_size: int = 32
    fallback_batch_size: int = 16
    epochs: int = 20
    serious_epochs: int = 80
    lr: float = 3e-4
    weight_decay: float = 0.05
    warmup_epochs: int = 5
    lambda_ce: float = 0.5
    val_fraction: float = 0.15
    target_pattern_masks: bool = True
    vq_confidence_threshold: float = 0.55
    fast_dev_run: bool = bool(int(os.environ.get('FAST_DEV_RUN', '0')))
    max_fast_train_images: int = 128
    num_workers: int = 2

CFG = Config()
if CFG.fast_dev_run:
    CFG.epochs = 1
    CFG.batch_size = 8

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = WORKING / f'aquatic_mae_{CFG.version}'
OUT.mkdir(parents=True, exist_ok=True)
RUN_SUMMARY_PATH = OUT / 'run_summary.json'

progress = {
    'status': 'running',
    'stage': 'start',
    'config': asdict(CFG),
    'device': str(DEVICE),
    'artifacts': [],
    'warnings': [],
}

def save_progress():
    RUN_SUMMARY_PATH.write_text(json.dumps(progress, indent=2, default=str), encoding='utf-8')

save_progress()
print({'device': str(DEVICE), 'out': str(OUT), 'fast_dev_run': CFG.fast_dev_run})


## Resolve dataset paths

The Kaggle mount can be nested, so find the directory that contains the required competition files instead of assuming one fixed path.


In [ ]:

REQUIRED = {'train', 'test', 'target.csv', 'dino_vq.py', 'codebook.npy'}
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

def find_dataset_root(base=Path('/kaggle/input')):
    candidates = []
    if base.exists():
        for target in base.rglob('target.csv'):
            root = target.parent
            names = {p.name for p in root.iterdir()}
            score = len(REQUIRED & names)
            if score >= 3:
                candidates.append((score, root))
    if not candidates:
        local = Path.cwd()
        names = {p.name for p in local.iterdir()}
        if len(REQUIRED & names) >= 3:
            return local
        raise FileNotFoundError('Could not find dataset root with target.csv/train/test/dino_vq.py/codebook.npy')
    candidates.sort(reverse=True, key=lambda item: item[0])
    return candidates[0][1]

progress['stage'] = 'resolve_paths'
save_progress()
DATA = find_dataset_root()
TRAIN_DIR = DATA / 'train'
TEST_DIR = DATA / 'test'
TARGET_CSV = DATA / 'target.csv'
DINO_VQ = DATA / 'dino_vq.py'
CODEBOOK = DATA / 'codebook.npy'

missing = [str(p) for p in [TRAIN_DIR, TEST_DIR, TARGET_CSV, DINO_VQ, CODEBOOK] if not p.exists()]
if missing:
    raise FileNotFoundError({'missing': missing, 'dataset_root': str(DATA)})

progress['dataset_root'] = str(DATA)
progress['stage'] = 'paths_ready'
save_progress()
print({'dataset_root': str(DATA)})


## Data validation and patch helpers

Patch order is row-major over the 8x8 grid. Visible-patch preservation is enforced by an assertion after every reconstruction paste.


In [ ]:

def list_images(folder):
    return sorted([p for p in folder.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES])

TRAIN_IMAGES = list_images(TRAIN_DIR)
TEST_IMAGES = list_images(TEST_DIR)
if CFG.fast_dev_run:
    TRAIN_IMAGES = TRAIN_IMAGES[:CFG.max_fast_train_images]

if not TRAIN_IMAGES:
    raise RuntimeError(f'No train images found in {TRAIN_DIR}')
if not TEST_IMAGES:
    raise RuntimeError(f'No test images found in {TEST_DIR}')

def build_image_index(paths):
    by_name = {p.name: p for p in paths}
    by_stem = {p.stem: p for p in paths}
    return by_name, by_stem

TEST_BY_NAME, TEST_BY_STEM = build_image_index(TEST_IMAGES)
TRAIN_BY_NAME, TRAIN_BY_STEM = build_image_index(TRAIN_IMAGES)

def resolve_image_path(image_id, by_name, by_stem):
    image_id = str(image_id)
    if image_id in by_name:
        return by_name[image_id]
    stem = Path(image_id).stem
    if stem in by_stem:
        return by_stem[stem]
    for suffix in IMAGE_SUFFIXES:
        name = image_id + suffix
        if name in by_name:
            return by_name[name]
    raise KeyError(f'Image id not found: {image_id}')

def parse_target_id(value):
    left, right = str(value).rsplit('_', 1)
    return left, int(right)

target_df = pd.read_csv(TARGET_CSV)
if list(target_df.columns)[:1] != ['Id']:
    raise ValueError(f'target.csv must contain Id column, got {list(target_df.columns)}')
target_df[['image_id', 'patch_index']] = target_df['Id'].apply(lambda x: pd.Series(parse_target_id(x)))
if not target_df['patch_index'].between(0, 63).all():
    raise ValueError('target.csv contains patch_index outside 0..63')

def load_image(path):
    img = Image.open(path).convert('RGB')
    if img.size != (CFG.image_size, CFG.image_size):
        raise ValueError(f'Expected 256x256 image, got {img.size}: {path}')
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

def save_image_tensor(tensor, path):
    arr = tensor.detach().clamp(0, 1).cpu().permute(1, 2, 0).numpy()
    # Save PNG bytes even if the original filename has a lossy suffix.
    Image.fromarray((arr * 255.0 + 0.5).astype(np.uint8)).save(path, format='PNG')

def patchify(imgs):
    p = CFG.patch_size
    b, c, h, w = imgs.shape
    assert h == CFG.image_size and w == CFG.image_size and c == 3
    x = imgs.reshape(b, c, 8, p, 8, p).permute(0, 2, 4, 3, 5, 1)
    return x.reshape(b, 64, p * p * c)

def unpatchify(patches):
    p = CFG.patch_size
    b = patches.shape[0]
    x = patches.reshape(b, 8, 8, p, p, 3).permute(0, 5, 1, 3, 2, 4)
    return x.reshape(b, 3, CFG.image_size, CFG.image_size)

def mask_from_indices(indices, device=None):
    mask = torch.zeros(64, dtype=torch.bool, device=device)
    mask[list(map(int, indices))] = True
    return mask

def preserve_visible(original, reconstructed, mask):
    orig_p = patchify(original.unsqueeze(0))[0]
    rec_p = patchify(reconstructed.unsqueeze(0))[0]
    rec_p[~mask] = orig_p[~mask]
    out = unpatchify(rec_p.unsqueeze(0))[0]
    check = patchify(out.unsqueeze(0))[0]
    if not torch.equal((check[~mask] * 255).round().byte(), (orig_p[~mask] * 255).round().byte()):
        raise AssertionError('Visible patches changed during preservation')
    return out

progress.update({
    'stage': 'data_validated',
    'train_images': len(TRAIN_IMAGES),
    'test_images': len(TEST_IMAGES),
    'target_rows': int(len(target_df)),
})
save_progress()
print({'train': len(TRAIN_IMAGES), 'test': len(TEST_IMAGES), 'target_rows': len(target_df)})


## Dataset, dataloader, and synthetic masks

Synthetic masks mimic the competition by hiding 8 to 20 patches. Optional target-pattern masks reuse patch layouts from `target.csv`.


In [ ]:

def target_patterns_from_csv(df):
    patterns = []
    for _, group in df.groupby('image_id'):
        idx = sorted(group['patch_index'].astype(int).tolist())
        if CFG.min_masked <= len(idx) <= CFG.max_masked:
            patterns.append(idx)
    return patterns

TARGET_PATTERNS = target_patterns_from_csv(target_df) if CFG.target_pattern_masks else []

class AquaticTrainDataset(Dataset):
    def __init__(self, image_paths, vq_cache=None):
        self.image_paths = list(image_paths)
        self.vq_cache = vq_cache or {}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = load_image(path)
        labels = self.vq_cache.get(path.name) or self.vq_cache.get(path.stem)
        if labels is None:
            labels = torch.full((64,), -100, dtype=torch.long)
        else:
            labels = torch.as_tensor(labels, dtype=torch.long)
        return {'image': image, 'name': path.name, 'vq_labels': labels}

class MaskSampler:
    def __init__(self, patterns=None):
        self.patterns = patterns or []

    def sample(self, batch_size, device):
        masks = torch.zeros(batch_size, 64, dtype=torch.bool, device=device)
        for b in range(batch_size):
            use_pattern = self.patterns and random.random() < 0.5
            if use_pattern:
                idx = random.choice(self.patterns)
            else:
                k = random.randint(CFG.min_masked, CFG.max_masked)
                idx = random.sample(range(64), k)
            masks[b, idx] = True
        return masks

rng = np.random.default_rng(CFG.seed)
indices = np.arange(len(TRAIN_IMAGES))
rng.shuffle(indices)
val_count = max(1, int(len(indices) * CFG.val_fraction))
val_idx = set(indices[:val_count].tolist())
train_paths = [p for i, p in enumerate(TRAIN_IMAGES) if i not in val_idx]
val_paths = [p for i, p in enumerate(TRAIN_IMAGES) if i in val_idx]
if not train_paths:
    train_paths, val_paths = TRAIN_IMAGES, TRAIN_IMAGES[:1]

mask_sampler = MaskSampler(TARGET_PATTERNS)
print({'train_split': len(train_paths), 'val_split': len(val_paths), 'target_patterns': len(TARGET_PATTERNS)})


## Optional VQ label cache

Try to use `dino_vq.py` for train/val labels. If its interface is not callable from this notebook, continue RGB-only and record the warning.


In [ ]:

def read_patch_csv(path):
    df = pd.read_csv(path)
    if not {'Id', 'Target'} <= set(df.columns):
        return None
    cache = {}
    for _, row in df.iterrows():
        image_id, patch_idx = parse_target_id(row['Id'])
        labels = cache.setdefault(image_id, [-100] * 64)
        labels[int(patch_idx)] = int(row['Target'])
    return {k: torch.tensor(v, dtype=torch.long) for k, v in cache.items()}

def try_run_dino_vq(image_dir, out_csv):
    candidates = [
        [sys.executable, str(DINO_VQ), '--image_dir', str(image_dir), '--output_csv', str(out_csv)],
        [sys.executable, str(DINO_VQ), '--input_dir', str(image_dir), '--output', str(out_csv)],
        [sys.executable, str(DINO_VQ), str(image_dir), str(out_csv)],
    ]
    errors = []
    for cmd in candidates:
        try:
            result = subprocess.run(cmd, cwd=str(DATA), text=True, capture_output=True, timeout=1800)
            if result.returncode == 0 and out_csv.exists():
                parsed = read_patch_csv(out_csv)
                if parsed:
                    return parsed
            errors.append({'cmd': cmd, 'returncode': result.returncode, 'stderr': result.stderr[-800:]})
        except Exception as exc:
            errors.append({'cmd': cmd, 'error': repr(exc)})
    raise RuntimeError(errors)

vq_cache = {}
try:
    progress['stage'] = 'vq_cache'
    save_progress()
    # This may be expensive; skip on FAST_DEV_RUN unless user explicitly asks with CACHE_VQ=1.
    if CFG.fast_dev_run and os.environ.get('CACHE_VQ', '0') != '1':
        raise RuntimeError('Skipping VQ cache in FAST_DEV_RUN')
    cache_csv = OUT / 'train_vq_cache.csv'
    vq_cache = try_run_dino_vq(TRAIN_DIR, cache_csv)
    progress['artifacts'].append(str(cache_csv))
    progress['vq_cache_images'] = len(vq_cache)
except Exception as exc:
    progress['warnings'].append(f'VQ cache disabled: {repr(exc)[:500]}')
    vq_cache = {}
finally:
    save_progress()

train_ds = AquaticTrainDataset(train_paths, vq_cache)
val_ds = AquaticTrainDataset(val_paths, vq_cache)
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
print({'vq_cache_images': len(vq_cache), 'train_batches': len(train_loader), 'val_batches': len(val_loader)})


## Baselines

Neighbor mean baseline fills each hidden patch from visible neighboring patches. It is cheap and useful for sanity checks.


In [ ]:

def neighbor_mean_fill(image, mask):
    patches = patchify(image.unsqueeze(0))[0]
    out = patches.clone()
    for idx in torch.where(mask)[0].tolist():
        r, c = divmod(idx, 8)
        neigh = []
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            rr, cc = r + dr, c + dc
            jj = rr * 8 + cc
            if 0 <= rr < 8 and 0 <= cc < 8 and not bool(mask[jj]):
                neigh.append(patches[jj])
        if neigh:
            out[idx] = torch.stack(neigh).mean(0)
        else:
            out[idx] = patches[~mask].mean(0)
    filled = unpatchify(out.unsqueeze(0))[0]
    return preserve_visible(image, filled, mask)

# tiny self-check
_sample = torch.rand(1, 3, 256, 256)
_p = patchify(_sample)
assert _p.shape == (1, 64, 3072)
assert torch.allclose(unpatchify(_p), _sample)
print('patch helpers ok')


## MAE model

Encoder sees visible patches only. Decoder receives encoded visible tokens plus learned mask tokens and predicts both RGB patch vectors and optional VQ logits.


In [ ]:

class SmallPatchMAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.patch_embed = nn.Linear(cfg.patch_dim, cfg.embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, cfg.num_patches, cfg.embed_dim))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.embed_dim,
            nhead=cfg.encoder_heads,
            dim_feedforward=cfg.embed_dim * 4,
            dropout=0.0,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.encoder_depth)
        self.enc_to_dec = nn.Linear(cfg.embed_dim, cfg.decoder_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, cfg.decoder_dim))
        self.dec_pos_embed = nn.Parameter(torch.zeros(1, cfg.num_patches, cfg.decoder_dim))
        dec_layer = nn.TransformerEncoderLayer(
            d_model=cfg.decoder_dim,
            nhead=cfg.decoder_heads,
            dim_feedforward=cfg.decoder_dim * 4,
            dropout=0.0,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerEncoder(dec_layer, num_layers=cfg.decoder_depth)
        self.rgb_head = nn.Linear(cfg.decoder_dim, cfg.patch_dim)
        self.vq_head = nn.Linear(cfg.decoder_dim, cfg.vq_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.dec_pos_embed, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, imgs, mask):
        patches = patchify(imgs)
        x = self.patch_embed(patches) + self.pos_embed
        b, n, d = x.shape

        visible = []
        lengths = []
        for i in range(b):
            tok = x[i, ~mask[i]]
            visible.append(tok)
            lengths.append(tok.shape[0])
        max_len = max(lengths)
        padded = x.new_zeros(b, max_len, d)
        pad_mask = torch.ones(b, max_len, dtype=torch.bool, device=x.device)
        for i, tok in enumerate(visible):
            padded[i, :tok.shape[0]] = tok
            pad_mask[i, :tok.shape[0]] = False

        encoded = self.encoder(padded, src_key_padding_mask=pad_mask)
        encoded = self.enc_to_dec(encoded)

        dec_tokens = self.mask_token.expand(b, n, -1).clone()
        for i in range(b):
            dec_tokens[i, ~mask[i]] = encoded[i, :lengths[i]]
        dec_tokens = dec_tokens + self.dec_pos_embed
        dec = self.decoder(dec_tokens)
        rgb = self.rgb_head(dec).sigmoid()
        vq_logits = self.vq_head(dec)
        return {'rgb_patches': rgb, 'vq_logits': vq_logits, 'target_patches': patches}

model = SmallPatchMAE(CFG).to(DEVICE)
param_count = sum(p.numel() for p in model.parameters())
progress['model_params'] = int(param_count)
save_progress()
print({'params_m': round(param_count / 1e6, 3)})


## Training

Use AMP on GPU, AdamW, cosine decay, and checkpoint both best validation loss and last epoch.


In [ ]:

def masked_l1(pred, target, mask):
    return F.l1_loss(pred[mask], target[mask])

def masked_ce(logits, labels, mask):
    valid = mask & (labels >= 0)
    if valid.sum() == 0:
        return None
    return F.cross_entropy(logits[valid], labels[valid])

def lr_for_epoch(epoch, steps_per_epoch, step):
    total_steps = max(1, CFG.epochs * steps_per_epoch)
    warmup_steps = max(1, CFG.warmup_epochs * steps_per_epoch)
    current = epoch * steps_per_epoch + step + 1
    if current <= warmup_steps:
        return CFG.lr * current / warmup_steps
    pct = (current - warmup_steps) / max(1, total_steps - warmup_steps)
    return CFG.lr * 0.5 * (1 + math.cos(math.pi * pct))

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
best_val = float('inf')
history = []

def run_epoch(loader, train=True, epoch=0):
    model.train(train)
    totals = {'loss': 0.0, 'rgb': 0.0, 'ce': 0.0, 'batches': 0, 'ce_batches': 0}
    for step, batch in enumerate(loader):
        imgs = batch['image'].to(DEVICE, non_blocking=True)
        labels = batch['vq_labels'].to(DEVICE, non_blocking=True)
        mask = mask_sampler.sample(imgs.shape[0], DEVICE)
        if train:
            for group in optimizer.param_groups:
                group['lr'] = lr_for_epoch(epoch, len(loader), step)
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            out = model(imgs, mask)
            rgb_loss = masked_l1(out['rgb_patches'], out['target_patches'], mask)
            ce_loss = masked_ce(out['vq_logits'], labels, mask)
            loss = rgb_loss if ce_loss is None else rgb_loss + CFG.lambda_ce * ce_loss
        if train:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        totals['loss'] += float(loss.detach().cpu())
        totals['rgb'] += float(rgb_loss.detach().cpu())
        if ce_loss is not None:
            totals['ce'] += float(ce_loss.detach().cpu())
            totals['ce_batches'] += 1
        totals['batches'] += 1
        if CFG.fast_dev_run and step >= 1:
            break
    denom = max(1, totals['batches'])
    return {
        'loss': totals['loss'] / denom,
        'rgb': totals['rgb'] / denom,
        'ce': totals['ce'] / max(1, totals['ce_batches']),
        'ce_batches': totals['ce_batches'],
    }

progress['stage'] = 'training'
save_progress()
try:
    for epoch in range(CFG.epochs):
        train_metrics = run_epoch(train_loader, train=True, epoch=epoch)
        val_metrics = run_epoch(val_loader, train=False, epoch=epoch)
        row = {'epoch': epoch + 1, 'train': train_metrics, 'val': val_metrics}
        history.append(row)
        print(row)
        torch.save({'model': model.state_dict(), 'config': asdict(CFG), 'history': history}, OUT / 'last.pt')
        if val_metrics['loss'] < best_val:
            best_val = val_metrics['loss']
            torch.save({'model': model.state_dict(), 'config': asdict(CFG), 'history': history}, OUT / 'best.pt')
        progress['history'] = history
        progress['best_val_loss'] = best_val
        progress['artifacts'] = sorted(set(progress['artifacts'] + [str(OUT / 'last.pt'), str(OUT / 'best.pt')]))
        save_progress()
except RuntimeError as exc:
    if 'out of memory' in str(exc).lower() and CFG.batch_size != CFG.fallback_batch_size:
        progress['warnings'].append('OOM hit; rerun with batch_size=16 or set FAST_DEV_RUN=1 for smoke')
        save_progress()
        raise
    raise


## Offline validation and visualizations

Validate on synthetic masks, paste visible patches back, and save a few mosaics for inspection.


In [ ]:

def make_mosaic(items, path, titles=None):
    if plt is None or not items:
        return
    cols = len(items)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
    if cols == 1:
        axes = [axes]
    for ax, img, title in zip(axes, items, titles or [''] * cols):
        arr = img.detach().clamp(0, 1).cpu().permute(1, 2, 0).numpy()
        ax.imshow(arr)
        ax.set_title(title)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)

@torch.no_grad()
def reconstruct_batch(imgs, mask):
    out = model(imgs, mask)
    rec = unpatchify(out['rgb_patches'])
    preserved = []
    for i in range(imgs.shape[0]):
        preserved.append(preserve_visible(imgs[i].cpu(), rec[i].cpu(), mask[i].cpu()))
    return torch.stack(preserved), out

progress['stage'] = 'offline_validation'
save_progress()
model.eval()
val_batch = next(iter(val_loader))
imgs = val_batch['image'].to(DEVICE)
mask = mask_sampler.sample(imgs.shape[0], DEVICE)
preserved, val_out = reconstruct_batch(imgs, mask)
masked_imgs = []
for i in range(min(4, imgs.shape[0])):
    patches = patchify(imgs[i:i+1].cpu())[0]
    patches[mask[i].cpu()] = 0
    masked_imgs.append(unpatchify(patches.unsqueeze(0))[0])
    mosaic_path = OUT / f'val_mosaic_{i}.png'
    make_mosaic([masked_imgs[-1], preserved[i], imgs[i].cpu()], mosaic_path, ['masked', 'reconstructed_preserved', 'ground_truth'])
    progress['artifacts'].append(str(mosaic_path))

progress['stage'] = 'offline_validation_done'
save_progress()
print({'validation_mosaics': len([p for p in OUT.glob('val_mosaic_*.png')])})


## Test inference

Read `target.csv`, reconstruct only images with required patches, preserve all visible patches, and write reconstructed images.


In [ ]:

progress['stage'] = 'test_inference'
save_progress()
ckpt_path = OUT / 'best.pt'
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
model.eval()

# Final validation command expects this exact relative folder name.
GENERATE_DIR = WORKING / 'generate'
GENERATE_DIR.mkdir(parents=True, exist_ok=True)
submission_rows = []
vq_candidate_rows = []

@torch.no_grad()
def infer_one(image_id, patch_indices):
    path = resolve_image_path(image_id, TEST_BY_NAME, TEST_BY_STEM)
    img = load_image(path)
    mask = mask_from_indices(patch_indices, device=DEVICE)
    rec, out = reconstruct_batch(img.unsqueeze(0).to(DEVICE), mask.unsqueeze(0))
    rec_img = rec[0]
    save_path = GENERATE_DIR / path.name
    save_image_tensor(rec_img, save_path)

    logits = out['vq_logits'][0].detach().softmax(-1).cpu()
    conf, pred = logits.max(-1)
    for patch_idx in patch_indices:
        vq_candidate_rows.append({
            'Id': f'{image_id}_{int(patch_idx)}',
            'Target': int(pred[int(patch_idx)]),
            'confidence': float(conf[int(patch_idx)]),
        })
    return save_path

for image_id, group in target_df.groupby('image_id'):
    infer_one(image_id, group['patch_index'].astype(int).tolist())

vq_candidate_df = pd.DataFrame(vq_candidate_rows)
vq_candidate_path = OUT / 'vq_head_candidates.csv'
vq_candidate_df.to_csv(vq_candidate_path, index=False)
progress['artifacts'].append(str(vq_candidate_path))
progress['generated_images'] = len(list(GENERATE_DIR.glob('*')))
save_progress()
print({'generated_images': progress['generated_images'], 'vq_candidates': len(vq_candidate_df), 'images_root': str(GENERATE_DIR)})



## Submission writing

Run the final competition encoder exactly as required:

```bash
python dino_vq.py --images-root generate --target-csv target.csv --codebook-path codebook.npy --output-csv submission.csv --batch-size 8
```

The notebook runs it from `/kaggle/working` after copying `dino_vq.py`, `target.csv`, and `codebook.npy` there.


In [ ]:

def run_required_dino_submission():
    work_dino = WORKING / 'dino_vq.py'
    work_target = WORKING / 'target.csv'
    work_codebook = WORKING / 'codebook.npy'
    shutil.copy2(DINO_VQ, work_dino)
    shutil.copy2(TARGET_CSV, work_target)
    shutil.copy2(CODEBOOK, work_codebook)

    cmd = [
        sys.executable,
        'dino_vq.py',
        '--images-root',
        'generate',
        '--target-csv',
        'target.csv',
        '--codebook-path',
        'codebook.npy',
        '--output-csv',
        'submission.csv',
        '--batch-size',
        '8',
    ]
    result = subprocess.run(cmd, cwd=str(WORKING), text=True, capture_output=True, timeout=7200)
    progress['dino_vq_command'] = ' '.join(['python'] + cmd[2:])
    progress['dino_vq_stdout_tail'] = result.stdout[-2000:]
    progress['dino_vq_stderr_tail'] = result.stderr[-2000:]
    if result.returncode != 0:
        raise RuntimeError({'returncode': result.returncode, 'stderr': result.stderr[-2000:]})
    submission_csv = WORKING / 'submission.csv'
    if not submission_csv.exists():
        raise FileNotFoundError(str(submission_csv))
    df = pd.read_csv(submission_csv)
    if list(df.columns) != ['Id', 'Target']:
        raise AssertionError(f'dino_vq submission columns mismatch: {list(df.columns)}')
    return df

progress['stage'] = 'write_submission'
save_progress()
try:
    submission = run_required_dino_submission()
    source = 'required_dino_vq_command'
except Exception as exc:
    progress['warnings'].append(f'required dino_vq command failed: {repr(exc)[:500]}')
    # Keep a valid-shaped emergency CSV for debugging, but status records that dino_vq failed.
    conf_map = {row.Id: (int(row.Target), float(row.confidence)) for row in vq_candidate_df.itertuples(index=False)}
    final_rows = []
    for row_id in target_df['Id'].astype(str).tolist():
        if row_id in conf_map:
            target = conf_map[row_id][0]
        else:
            target = 0
            progress['warnings'].append(f'Zero fallback used for {row_id}')
        final_rows.append({'Id': row_id, 'Target': int(target)})
    submission = pd.DataFrame(final_rows, columns=['Id', 'Target'])
    source = 'fallback_vq_or_zero_after_dino_failure'

if len(submission) != len(target_df):
    raise AssertionError(f'submission row count mismatch: {len(submission)} vs {len(target_df)}')
if list(submission.columns) != ['Id', 'Target']:
    raise AssertionError('submission columns mismatch')
if not submission['Target'].between(0, 1023).all():
    raise AssertionError('submission Target outside 0..1023')

submission_path = WORKING / 'submission_v1.csv'
submission.to_csv(submission_path, index=False)
local_submission_path = OUT / 'submission_v1.csv'
submission.to_csv(local_submission_path, index=False)
progress['submission_source'] = source
progress['submission_rows'] = int(len(submission))
progress['artifacts'] = sorted(set(progress['artifacts'] + [
    str(WORKING / 'submission.csv'),
    str(submission_path),
    str(local_submission_path),
    str(GENERATE_DIR),
]))
save_progress()
print({'submission': str(submission_path), 'rows': len(submission), 'source': source})


## Final summary and output bundle

Write final metadata and a compact ZIP so outputs are easy to download on Windows.


In [ ]:

progress['stage'] = 'bundle_outputs'
save_progress()
zip_path = WORKING / 'aquatic_mae_v1_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(OUT.parent))
    for name in ['submission.csv', 'submission_v1.csv']:
        sub = WORKING / name
        if sub.exists():
            zf.write(sub, sub.name)

progress['status'] = 'complete' if progress.get('submission_source') == 'required_dino_vq_command' else 'complete_with_warnings'
progress['stage'] = 'done'
progress['artifacts'] = sorted(set(progress['artifacts'] + [str(zip_path), str(RUN_SUMMARY_PATH)]))
save_progress()
print(json.dumps({
    'status': progress['status'],
    'submission_rows': progress.get('submission_rows'),
    'submission_source': progress.get('submission_source'),
    'dino_vq_command': progress.get('dino_vq_command'),
    'warnings': progress.get('warnings', [])[:10],
    'zip': str(zip_path),
}, indent=2))
